In [ ]:
#idea: create a way to simulate success of your portfolio based on simulations
#user inputs which stocks it is invested in, how much it is invested, and code outputs expected return over different timeframes
#could also make this compatable with indexes that automatically trade, such as market cap-based S&P 500,
#equal distribution S&P 500, mutual funds, etc.

#desired outputs:
#graph of 10 simulations of ur stock
#mean, std, and VaR for each stock over 1y, 5y, 10y, and 20y intervals

#this code is meant for just playing around with stock simulations

import sys
print(sys.version)

!pip install -q tidyfinance
!pip install -q adjustText

import pandas as pd
import numpy as np
import tidyfinance as tf
import matplotlib.pyplot as plt

from plotnine import *
from mizani.formatters import percent_format
from adjustText import adjust_text

x = 'continue'
symbols = []
while x != 'no':
  x = input('enter a stock symbol, or enter (no) to continue\n')
  if x != 'no':
    symbols.append(x)


prices_daily = tf.download_data(
    domain = 'stock_prices',
    symbols = symbols,
    start_date = '01-01-2000',
    end_date = '12-31-2025'
)

returns_daily = (prices_daily
    .assign(
        ret = lambda x: x.groupby('symbol')['adjusted_close'].pct_change()
    )
)

summary_ret = (returns_daily
    .groupby('symbol', as_index = False)
    .agg(
        mu = ('ret','mean'),
        sigma = ('ret','std')
    )
)
print('summary data for daily returns')
print(summary_ret)

def simulate_gbm(S0, annualized_mu, annualized_sigma, total_years, steps_per_year=252, n_sims=1000):
    dt = 1/252
    total_steps = total_years * steps_per_year
    Z = np.random.normal(0, 1, (total_steps, n_sims))
    returns = (annualized_mu - (1/2)*annualized_sigma**2) * dt + annualized_sigma * np.sqrt(dt) * Z
    log_paths = np.cumsum(returns, axis = 0)
    S = S0 * np.exp(log_paths)

    return S

S0 = 100

T = int(input('how many years would you like the simulation to run?\n'))
print('running 1000 simulations with starting value of $100')

simulation_results = {}

for index, row in summary_ret.iterrows():
    symbol = row['symbol']
    daily_mu = row['mu']
    daily_sigma = row['sigma']

    annualized_mu = daily_mu * 252
    annualized_sigma = daily_sigma * np.sqrt(252)

    paths = simulate_gbm(
        S0=S0, annualized_mu=annualized_mu, annualized_sigma=annualized_sigma, total_years=T
        )
    simulation_results[symbol] = paths

    print(f"\nSimulated Paths for {symbol}:")
    plt.figure(figsize=(10, 6))
    plt.plot(paths[:, :10])
    plt.title(f'Simulated Stock Paths for {symbol}')
    plt.xlabel('Days')
    plt.ylabel('Stock Price')
    plt.show()

def q05(x):
  return x.quantile(0.05)

all_results_path = pd.DataFrame()

for symbol, paths_data in simulation_results.items():
    final_simulated_prices = pd.Series(paths_data[251, 0:999])
    results = final_simulated_prices.agg(
        mu = 'mean',
        sigma = 'std',
        value_at_risk = q05
    ).to_frame().T
    results['symbol'] = symbol
    all_results_path = pd.concat([all_results_path, results], ignore_index=True)

print('Value at Risk calculated at 0.05 quintile')
print("\nSummary of all simulation results:")
print(all_results_path)